In [1]:
import cups
import os

In [2]:
class PrinterManager:
    def __init__(self):
        try:
            self.conn = cups.Connection()
        except Exception as e:
            raise RuntimeError(f"Impossible de se connecter à CUPS : {e}")

    def lister_imprimantes(self):
        """Retourne une liste des noms d'imprimantes disponibles."""
        try:
            printers = self.conn.getPrinters()
            return list(printers.keys())
        except Exception:
            return []

    def obtenir_statut(self, nom_imprimante):
        """
        CORRECTION : Utilise getPrinters() pour récupérer les attributs.
        Retourne un dictionnaire avec l'état précis.
        """
        try:
            # On récupère toutes les imprimantes et on sélectionne la bonne
            printers = self.conn.getPrinters()
            
            if nom_imprimante not in printers:
                return {"en_erreur": True, "message": "Imprimante introuvable", "bloquee": False}

            attrs = printers[nom_imprimante]
            
            # Récupération des attributs spécifiques
            # printer-state-reasons est une liste de chaînes (ex: ['media-empty-warning'])
            reasons = attrs.get('printer-state-reasons', ['none'])
            state = attrs.get('printer-state', 3) # 3=Idle, 4=Processing, 5=Stopped
            
            status = {
                "nom": nom_imprimante,
                "en_erreur": False,
                "message": "Prête",
                "bloquee": False,
                "reasons": reasons
            }

            # Conversion en une seule chaîne pour la recherche facile
            reasons_str = " ".join(reasons).lower()

            # Logique de détection d'erreurs
            if state == 5: # Stopped
                status["bloquee"] = True
                status["message"] = "En pause (File stoppée)."

            if "media-empty" in reasons_str or "media-needed" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Plus de papier."
            elif "marker-supply-empty" in reasons_str or "no-toner" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Plus d'encre."
            elif "offline" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Imprimante déconnectée."
            elif "input-tray-missing" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Bac papier manquant."
            elif state == 4:
                status["message"] = "Impression en cours..."
            elif "none" not in reasons_str and reasons:
                 # Cas d'autres erreurs génériques
                 status["message"] = f"Info : {reasons_str}"

            return status

        except Exception as e:
            return {"en_erreur": True, "message": f"Erreur système : {e}", "bloquee": False}

    def relancer_file(self, nom_imprimante):
        """Réactive l'imprimante (nécessaire après une erreur papier sur Selphy)."""
        try:
            self.conn.enablePrinter(nom_imprimante)
            print(f"-> File d'attente de '{nom_imprimante}' réactivée.")
            return True
        except cups.IPPError:
            return False

    def imprimer_image(self, nom_imprimante, chemin_image, options=None):
        """Imprime une image."""
        if not os.path.exists(chemin_image):
            raise FileNotFoundError(f"Fichier introuvable : {chemin_image}")
        
        if options is None:
            # Options par défaut pour Canon Selphy CP1500
            options = {
                "fit-to-page": "True",
                # "media": "Postcard" # Décommentez si besoin de forcer le format
            }
            
        try:
            job_id = self.conn.printFile(nom_imprimante, chemin_image, "Job Python", options)
            return job_id
        except cups.IPPError as e:
            raise RuntimeError(f"Erreur d'envoi CUPS : {e}")

In [3]:
pm = PrinterManager()

In [4]:
imprimantes = pm.lister_imprimantes()
print(f"Imprimantes dispos : {imprimantes}")

Imprimantes dispos : ['_maison__Canon_G5000_series', 'Canon_SELPHY_CP1500']


In [28]:
mon_imprimante = "Canon_SELPHY_CP1500" # Remplacez par le nom exact trouvé
etat = pm.obtenir_statut(mon_imprimante)
print(etat)

{'nom': 'Canon_SELPHY_CP1500', 'en_erreur': True, 'message': 'ERREUR : Bac papier manquant.', 'bloquee': False, 'reasons': ['cups-waiting-for-job-completed', 'input-tray-missing']}


In [16]:

if mon_imprimante in imprimantes:
    # 2. Check avant impression
    etat = pm.obtenir_statut(mon_imprimante)
    print(f"Statut avant : {etat['message']}")

    if etat['bloquee']:
        print("Réactivation de la file...")
        pm.relancer_file(mon_imprimante)

    if not etat['en_erreur']:
        # 3. Impression
        try:
            jid = pm.imprimer_image(mon_imprimante, "photo.jpg")
            print(f"Impression lancée ! ID: {jid}")
        except Exception as e:
            print(e)
    else:
        print("Corrigez l'erreur avant d'imprimer.")
else:
    print(f"Imprimante '{mon_imprimante}' non trouvée. Vérifiez le nom et la connexion.")

Statut avant : Prête
Impression lancée ! ID: 13


In [29]:
pm.obtenir_statut(mon_imprimante)

{'nom': 'Canon_SELPHY_CP1500',
 'en_erreur': True,
 'message': 'ERREUR : Bac papier manquant.',
 'bloquee': False,
 'reasons': ['cups-waiting-for-job-completed', 'input-tray-missing']}